In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [3]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Xie2021_Part1.h5ad")

In [4]:
adata = adata[adata.obs['donor_id'].isin(["Patient_1", "Patient_2", "Patient_3", "Patient_4"])]

In [5]:
df_obs = pd.DataFrame(adata.obs)

In [6]:
del adata.obs

In [7]:
adata = adata.raw.to_adata()

In [8]:
adata

AnnData object with n_obs × n_vars = 94066 × 16040
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [9]:
#adata = adata.raw.to_adata()

In [10]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 4066/4066 [00:04<00:00, 817.45it/s]


In [15]:
adata.X = X_counts_recovered

In [16]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [17]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [18]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [19]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [20]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF367,False,2160,2.296260,True,0.011465,0.008091,0.739362
SULT1B1,False,2050,2.179321,True,0.014377,0.015259,1.461912
HDHD2,False,6245,6.638956,True,0.029579,0.018951,0.693298
MORF4L2-AS1,False,1443,1.534029,True,0.007473,0.005008,0.716224
TMEM53,False,2077,2.208024,True,0.009150,0.005950,0.720609
...,...,...,...,...,...,...,...
TPTEP2-CSNK1E,False,779,0.828142,True,0.003888,0.002640,0.727302
C13orf46,False,304,0.323177,True,0.001507,0.001037,0.743585
CFAP97D2,False,98,0.104182,True,0.000484,0.000320,0.675563


In [21]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [22]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [23]:
adata.var = df_tmp

In [24]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [25]:
adata

View of AnnData object with n_obs × n_vars = 94066 × 14915
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [26]:
adata.obs['donor_id'] = df_obs['donor_id']

In [28]:
metadata_data = {
    'Author': ['Xie2021'] * 4,
    'donor_id': ["Patient_1", "Patient_2", "Patient_3", "Patient_4"],
    'stage': ['Primary'] * 4,
    'assay': ['10x 3\' v3'] * 4,
    'tissue': ['right parietal lobe', 'left temporal lobe', 'right occipital lobe', 'right frontal lobe'],
    'Cells': ['Total'] * 4,
    'Method': ['cell'] * 4
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

    Author   donor_id    stage      assay                tissue  Cells Method
0  Xie2021  Patient_1  Primary  10x 3' v3   right parietal lobe  Total   cell
1  Xie2021  Patient_2  Primary  10x 3' v3    left temporal lobe  Total   cell
2  Xie2021  Patient_3  Primary  10x 3' v3  right occipital lobe  Total   cell
3  Xie2021  Patient_4  Primary  10x 3' v3    right frontal lobe  Total   cell


In [29]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

        donor_id   Author    stage      assay               tissue  Cells  \
0      Patient_1  Xie2021  Primary  10x 3' v3  right parietal lobe  Total   
1      Patient_1  Xie2021  Primary  10x 3' v3  right parietal lobe  Total   
2      Patient_1  Xie2021  Primary  10x 3' v3  right parietal lobe  Total   
3      Patient_1  Xie2021  Primary  10x 3' v3  right parietal lobe  Total   
4      Patient_1  Xie2021  Primary  10x 3' v3  right parietal lobe  Total   
...          ...      ...      ...        ...                  ...    ...   
94061  Patient_4  Xie2021  Primary  10x 3' v3   right frontal lobe  Total   
94062  Patient_4  Xie2021  Primary  10x 3' v3   right frontal lobe  Total   
94063  Patient_4  Xie2021  Primary  10x 3' v3   right frontal lobe  Total   
94064  Patient_4  Xie2021  Primary  10x 3' v3   right frontal lobe  Total   
94065  Patient_4  Xie2021  Primary  10x 3' v3   right frontal lobe  Total   

      Method  
0       cell  
1       cell  
2       cell  
3       cell  


In [30]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [31]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
tumour1_AAACCCAAGCTCTTCC-1-0-1,Patient_1,2189,1950.340210,Non-neoplastic,Vascular,Mural cell,Pericyte,Pericytes,mural cell
tumour1_AAACCCAAGGACAGTC-1-0-1,Patient_1,894,1347.239014,Non-neoplastic,Myeloid,TAM-BDM,Unknown,Unknown,macrophage
tumour1_AAACCCAAGGGTTTCT-1-0-1,Patient_1,674,1182.380859,Non-neoplastic,Myeloid,TAM-BDM,Unknown,Unknown,macrophage
tumour1_AAACCCAAGTAGTCAA-1-0-1,Patient_1,1251,1551.636108,Non-neoplastic,Myeloid,TAM-BDM,Macrophage,Pluripotent Stem Cells,macrophage
tumour1_AAACCCAAGTCACAGG-1-0-1,Patient_1,1512,1647.535522,Non-neoplastic,Myeloid,TAM-BDM,Endothelial cell,Endothelial Cells,macrophage
...,...,...,...,...,...,...,...,...,...
control4_TTTGTTGTCGATTGGT-1-0-1,Patient_4,1546,1652.489258,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Microglia,macrophage
control4_TTTGTTGTCGCCAATA-1-0-1,Patient_4,990,1458.386353,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
control4_TTTGTTGTCTCGACCT-1-0-1,Patient_4,1110,1499.193604,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Microglia,macrophage
control4_TTTGTTGTCTGGGCCA-1-0-1,Patient_4,848,1375.870605,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Microglia,macrophage


In [32]:
merged_obs_df.index= df_obs.index

In [33]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [34]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [35]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [36]:
del merged_obs_df['donor_id_y']

In [37]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [38]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [39]:
adata.obs = merged_obs_df

In [40]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    684 total control genes are used. (0:00:03)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    812 total control genes are used. (0:00:04)
-->     'phase', cell cycle phase (adata.obs)


In [41]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Xie2021_Part3.h5ad")